<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/locus_id_converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Convert RefSeq transcript IDs to RAP-DB locus IDs
g:Profiler's rice database uses RAP-DB/Ensembl Plants locus IDs (e.g. `Os01g0143400`) as its native ID space, not NCBI RefSeq accessions (`XM_...`). This fetches the GenBank record for each transcript and pulls out its `/gene=` (locus) tag, so you can submit locus IDs to g:Profiler instead.

In [ ]:
# Cell 1: install dependencies
!pip install -q biopython pandas requests

In [ ]:
# Cell 2: upload your hit_counts_summary_rice.csv
from google.colab import files
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
print('Uploaded:', INPUT_FILE)

In [ ]:
# Cell 3: config
import os, re, time
import requests
import pandas as pd

TRANSCRIPT_COL = 'transcript_id'
OUTPUT_FILE = 'chilli_hit_counts_with_locus.csv'
LOCUS_IDS_TXT = 'chilli_locus_ids.txt'
CHECKPOINT_FILE = 'locus_lookup_checkpoint.csv'

# Chilli pepper. Use taxid 4071 if you want the whole Capsicum genus.
ORGANISM = 'Capsicum annuum'
TAXID = '4072'

EMAIL = 'anjali.d@stringbio.com'    # required by NCBI
API_KEY = '4b6a3c9a2224f25ec40bc9ecec1efb283809'

EUTILS_BASE = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'
HEADERS = {'User-Agent': 'chilli-locus-lookup/1.0', 'Connection': 'close', 'Accept-Encoding': 'identity'}

BATCH_SIZE = 20
CHECKPOINT_EVERY = 5

In [ ]:
# Cell 4: helper functions

def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]


def _post_with_retries(url, data, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(url, data=data, headers=HEADERS, timeout=60)
            r.raise_for_status()
            return r
        except Exception as e:
            wait = 2 ** attempt
            print(f'    [request failed, attempt {attempt}/{max_retries}]: {e} -- retrying in {wait}s')
            time.sleep(wait)
    return None


def batch_transcript_to_locus(transcript_ids):
    """Fetch GenBank records for a batch of transcript IDs and pull the /gene= locus tag,
    plus /locus_tag= as a fallback."""
    result = {tid: None for tid in transcript_ids}

    efetch_data = {
        'db': 'nuccore',
        'id': transcript_ids,
        'rettype': 'gb',
        'retmode': 'text',
        'api_key': API_KEY,
        'email': EMAIL,
    }
    r = _post_with_retries(f'{EUTILS_BASE}/efetch.fcgi', efetch_data)
    if r is None:
        print('  [efetch batch failed after retries, skipping this batch]')
        return result

    text = r.text
    records = text.split('\n//\n')

    for rec in records:
        if not rec.strip():
            continue
        ver_match = re.search(r'^VERSION\s+(\S+)', rec, re.MULTILINE)
        if not ver_match:
            continue
        tid = ver_match.group(1)
        if tid not in result:
            continue

        gene_match = re.search(r'/gene="([^"]+)"', rec)
        locus_tag_match = re.search(r'/locus_tag="([^"]+)"', rec)

        if gene_match:
            result[tid] = gene_match.group(1)
        elif locus_tag_match:
            result[tid] = locus_tag_match.group(1)

    return result

In [ ]:
# Cell 5: load data + resume support
df = pd.read_csv(INPUT_FILE)
df.columns = [c.strip() for c in df.columns]  # in case of stray whitespace in headers
all_transcripts = df[TRANSCRIPT_COL].dropna().unique().tolist()
print(f'Found {len(all_transcripts)} unique transcript IDs.')

if os.path.exists(CHECKPOINT_FILE):
    checkpoint_df = pd.read_csv(CHECKPOINT_FILE)
    done_transcripts = set(checkpoint_df[TRANSCRIPT_COL])
    transcript_to_locus = dict(zip(checkpoint_df[TRANSCRIPT_COL], checkpoint_df['locus_id']))
    print(f'Resuming: {len(done_transcripts)} transcripts already resolved.')
else:
    checkpoint_df = pd.DataFrame(columns=[TRANSCRIPT_COL, 'locus_id'])
    done_transcripts = set()
    transcript_to_locus = {}

remaining = [t for t in all_transcripts if t not in done_transcripts]
print(f'{len(remaining)} transcripts left to resolve.')

In [ ]:
# Cell 6: batched resolution with checkpointing
batches = list(chunked(remaining, BATCH_SIZE))
new_rows = []
for i, batch in enumerate(batches, 1):
    print(f'[batch {i}/{len(batches)}] resolving {len(batch)} transcripts...')
    batch_result = batch_transcript_to_locus(batch)
    transcript_to_locus.update(batch_result)
    for tid, locus in batch_result.items():
        new_rows.append({TRANSCRIPT_COL: tid, 'locus_id': locus})

    if i % CHECKPOINT_EVERY == 0 or i == len(batches):
        checkpoint_df = pd.concat([checkpoint_df, pd.DataFrame(new_rows)], ignore_index=True)
        checkpoint_df.to_csv(CHECKPOINT_FILE, index=False)
        new_rows = []
        print(f'  checkpoint saved ({len(checkpoint_df)} total resolved)')

resolved_count = sum(1 for v in transcript_to_locus.values() if v)
print(f'\nResolved {resolved_count}/{len(all_transcripts)} transcripts to a locus ID.')

In [ ]:
# Cell 7: merge back into the original data and save
df['locus_id'] = df[TRANSCRIPT_COL].map(transcript_to_locus)
df.to_csv(OUTPUT_FILE, index=False)

# also write a plain list of unique locus IDs, ready to paste into g:Profiler,
# ordered by hit_count (highest first) if that column exists
sort_col = 'hit_count' if 'hit_count' in df.columns else None
locus_df = df.dropna(subset=['locus_id'])
if sort_col:
    locus_df = locus_df.sort_values(sort_col, ascending=False)
unique_locus_ids = list(dict.fromkeys(locus_df['locus_id']))  # dedupe, preserve order

with open(LOCUS_IDS_TXT, 'w') as f:
    f.write('\n'.join(unique_locus_ids))

print(f'{len(unique_locus_ids)} unique locus IDs written to {LOCUS_IDS_TXT}')
print(f'Sample: {unique_locus_ids[:10]}')

In [ ]:
# Cell 8: download results
from google.colab import files
files.download(OUTPUT_FILE)
files.download(LOCUS_IDS_TXT)

# For making a custom gmt if your organism is not in g:Profiler's dropdown.
This is for cotton. Change it accordingly.

In [ ]:
# === CottonGen genes2Go (Excel) -> g:Profiler custom GMT ===
# Run this one cell in Colab. It will ask you to upload the genes2Go file.
!pip -q install openpyxl >/dev/null 2>&1

import gzip, io
import pandas as pd
from collections import defaultdict
from google.colab import files

# ---- SETTINGS ----
GENE_LEVEL = True          # True -> Ghir_A01G000010 | False -> keep isoform .1
OUT_GMT    = "ghirsutum_HAU_custom.gmt"
# ------------------

print("Upload the genes2Go file (.xlsx or .xlsx.gz):")
up = files.upload()
fname = list(up.keys())[0]
data  = up[fname]
data  = gzip.decompress(data) if fname.endswith(".gz") else data
df    = pd.read_excel(io.BytesIO(data))

df.columns = [str(c).strip() for c in df.columns]
cols = df.columns.tolist()
if {"Query", "Match", "Description"}.issubset(cols):
    qc, mc, dc = "Query", "Match", "Description"
else:
    qc, mc, dc = cols[0], cols[1], cols[2]          # positional fallback
    print("Using positional columns:", qc, mc, dc)
print("Columns:", cols, "| rows:", len(df))

def norm_id(q):
    core = str(q).split("_HAU")[0]                  # drop assembly tag
    return core.split(".")[0] if GENE_LEVEL else core

term2genes, term2desc = defaultdict(set), {}
for q, go, desc in zip(df[qc], df[mc], df[dc]):
    go = str(go).strip()
    if not go.startswith("GO:"):
        continue
    term2genes[go].add(norm_id(q))
    term2desc.setdefault(go, str(desc).strip())

with open(OUT_GMT, "w") as out:
    for go, genes in term2genes.items():
        out.write(f"{go}\t{term2desc[go]}\t" + "\t".join(sorted(genes)) + "\n")

all_genes = set().union(*term2genes.values()) if term2genes else set()
print(f"\nWrote {OUT_GMT}")
print(f"{len(term2genes)} GO terms | {len(all_genes)} unique genes")
if all_genes:
    print("Sample gene IDs:", sorted(all_genes)[:3])

files.download(OUT_GMT)

In [ ]:
# === LOC-native GO GMT for g:Profiler from NCBI gene2go (G. hirsutum, taxid 3635) ===
import os, gzip
from collections import defaultdict

TAXID   = "3635"
OUT_GMT = "ghirsutum_LOC_custom.gmt"
G2G     = "gene2go.gz"

if not os.path.exists(G2G):
    print("Downloading NCBI gene2go (~hundreds of MB, 1-3 min)...")
    !wget -q https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2go.gz -O gene2go.gz
    print("done.")

term2genes = defaultdict(set)
term2name  = {}
n = 0
with gzip.open(G2G, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.rstrip("\n").split("\t")
        if p[0] != TAXID:                 # tax_id
            continue
        gene_id, go_id, go_name, category = p[1], p[2], p[5], p[7]
        term2genes[go_id].add("LOC" + gene_id)
        term2name.setdefault(go_id, f"{category}:{go_name}")
        n += 1

if n == 0:
    print("\n*** No GO annotations for taxid 3635 in gene2go. ***")
    print("Use the fallback (LOC->Ghir positional mapping) instead — ask me for that cell.")
else:
    with open(OUT_GMT, "w") as out:
        for go, genes in term2genes.items():
            out.write(f"{go}\t{term2name[go]}\t" + "\t".join(sorted(genes)) + "\n")
    genes = set().union(*term2genes.values())
    print(f"\nannotations: {n}")
    print(f"GO terms:    {len(term2genes)}")
    print(f"genes (LOC): {len(genes)}")
    print("sample:", sorted(genes)[:3])
    from google.colab import files
    files.download(OUT_GMT)

In [ ]:
# === LOC-native GO GMT for g:Profiler — Capsicum annuum (taxid 4072) ===
import os, gzip
from collections import defaultdict

TAXID   = "4072"                          # C. annuum
OUT_GMT = "cannuum_LOC_custom.gmt"
G2G     = "gene2go.gz"

if not os.path.exists(G2G):
    print("Downloading NCBI gene2go (~hundreds of MB, 1-3 min)...")
    !wget -q https://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2go.gz -O gene2go.gz
    print("done.")

term2genes, term2name, n = defaultdict(set), {}, 0
with gzip.open(G2G, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.rstrip("\n").split("\t")
        if p[0] != TAXID:
            continue
        gid, go, name, cat = p[1], p[2], p[5], p[7]
        term2genes[go].add("LOC" + gid)
        term2name.setdefault(go, f"{cat}:{name}")
        n += 1

if n == 0:
    print("\n*** No GO annotations for taxid 4072 in gene2go — tell me and we'll use the fallback. ***")
else:
    with open(OUT_GMT, "w") as out:
        for go, genes in term2genes.items():
            out.write(f"{go}\t{term2name[go]}\t" + "\t".join(sorted(genes)) + "\n")
    genes = set().union(*term2genes.values())
    print(f"annotations: {n} | GO terms: {len(term2genes)} | genes: {len(genes)}")
    print("sample:", sorted(genes)[:3])
    from google.colab import files
    files.download(OUT_GMT)